# Ансамбль B с CatBoost и статистиками по таксонам

Логистическая регрессия, ExtraTrees и CatBoost усредняют оценки класса 1 с одинаковыми весами. Входы включают обнаружение mOTU, клинические метаданные, оральную сигнатуру и статистики по таксономическим группам.

Для запуска нужны NumPy, pandas, scikit-learn и CatBoost. Положите train.csv и test.csv рядом с ноутбуком или в папку data. При необходимости задайте пути через GUT_TRAIN и GUT_TEST.

In [1]:
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

DATA = Path("data") if Path("data/train.csv").is_file() else Path(".")
TRAIN = Path(os.environ.get("GUT_TRAIN", DATA / "train.csv"))
TEST = Path(os.environ.get("GUT_TEST", TRAIN.parent / "test.csv"))
SUBMISSION = Path(os.environ.get("GUT_SUBMISSION", "submission_oral_ensemble_B.csv"))


## Исходное разбиение

Воспроизводим фиксированное стратифицированное разбиение по когорте и классу. Для проверки загружаем микробные признаки только 332 участников обучающей части. Порядок участников и порядок столбцов совпадают с исходным экспериментом.

In [2]:
keys = pd.read_csv(TRAIN, usecols=["sample_id", "cohort", "label"], dtype={"sample_id": str})
keys = keys.sort_values("sample_id")
dev_rows, _ = train_test_split(
    keys.index.to_numpy(), test_size=0.2, random_state=42,
    stratify=keys.cohort + "/" + keys.label.astype(str),
)
keep = set(dev_rows)
dev = pd.read_csv(TRAIN, dtype={"sample_id": str}, skiprows=lambda i: i > 0 and i - 1 not in keep)
dev = dev.set_index("sample_id").loc[keys.loc[dev_rows, "sample_id"]].reset_index()
taxa = sorted(c for c in dev if re.fullmatch(r".+ \[(?:ref|meta)_mOTU_v2_\d+\]", c))
assert len(dev) == 332 and dev.sample_id.is_unique and len(taxa) == 849

## Признаки и модель

Четыре признака сигнатуры — число обнаруженных оральных mOTU, сумма долей, среднее значение ln(доля + 10⁻⁶) и максимальная доля. Три подвида *F. nucleatum* учитываются по отдельности. Медианы, масштабы и категории оцениваются заново внутри каждой обучающей части.

Для каждой группы добавлены максимум, минимум, сумма, среднее, стандартное отклонение и медиана исходных долей. Группы задаются точным равенством первого токена аннотации после снятия префикса неопределённости. Каждый mOTU входит ровно в одну группу. Для группы из одного mOTU стандартное отклонение равно нулю. Пропуски и масштабирование обрабатываются внутри обучающей части.

In [3]:
ORAL = [
    "Parvimonas micra [ref_mOTU_v2_1145]",
    "Gemella morbillorum [ref_mOTU_v2_4513]",
    "Fusobacterium nucleatum s. animalis [ref_mOTU_v2_0776]",
    "Fusobacterium nucleatum s. vincentii [ref_mOTU_v2_0754]",
    "Fusobacterium nucleatum s. nucleatum [ref_mOTU_v2_0777]",
    "Peptostreptococcus stomatis [ref_mOTU_v2_4614]",
    "Solobacterium moorei [ref_mOTU_v2_0531]",
    "Porphyromonas asaccharolytica [ref_mOTU_v2_1517]",
    "Prevotella intermedia [ref_mOTU_v2_0515]",
]

assert set(ORAL).issubset(taxa)

def detected(x):
    return (x > 0).astype(float)

def oral_features(x):
    return np.column_stack((
        (x > 0).sum(axis=1), x.sum(axis=1),
        np.log(x + 1e-6).mean(axis=1), x.max(axis=1),
    ))

# Используем первый токен нормализованной таксономической аннотации.
# У некоторых mOTU это род, у других доступна только более широкая группа.
def taxon_group(column):
    annotation = column.rsplit(" [", 1)[0]
    annotation = re.sub(r"^(?:(?:unknown|uncultured|cand\.)\s+)+", "", annotation,
                        flags=re.IGNORECASE)
    return annotation.split()[0]

group_members = {}
for i, column in enumerate(taxa):
    group_members.setdefault(taxon_group(column), []).append(i)
motu_groups = tuple((name, tuple(group_members[name])) for name in sorted(group_members))
assert sorted(i for _, indices in motu_groups for i in indices) == list(range(len(taxa)))
STAT_NAMES = ("max", "min", "sum", "mean", "std", "median")
stat_feature_names = [f"{name}_{stat}" for name, _ in motu_groups for stat in STAT_NAMES]

def group_statistics(x):
    x = np.asarray(x, dtype=float)
    columns = []
    for _, indices in motu_groups:
        block = x[:, indices]
        std = block.std(axis=1, ddof=1) if len(indices) > 1 else np.zeros(len(block))
        columns.extend((block.max(axis=1), block.min(axis=1), block.sum(axis=1),
                        block.mean(axis=1), std, np.median(block, axis=1)))
    return np.column_stack(columns)

def make_model():
    stats = make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        FunctionTransformer(group_statistics),
        StandardScaler(),
    )
    micro = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                          FunctionTransformer(detected), StandardScaler())
    numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    categories = make_pipeline(SimpleImputer(strategy="most_frequent"),
                               OneHotEncoder(handle_unknown="ignore"))
    oral = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                         FunctionTransformer(oral_features), StandardScaler())
    preprocess = ColumnTransformer([
        ("microbial", micro, taxa),
        ("numeric", numeric, ["age", "bmi", "library_size"]),
        ("category", categories, ["gender", "cohort"]),
        ("oral", oral, ORAL),
        ("group_stats", stats, taxa),
    ])
    ensemble = VotingClassifier(voting="soft", estimators=[
        ("logistic", LogisticRegression(C=0.01, solver="liblinear", max_iter=2000, random_state=42)),
        ("trees", ExtraTreesClassifier(n_estimators=160, max_features="sqrt", min_samples_leaf=2,
                                      random_state=42, n_jobs=1)),
        ("boosting", CatBoostClassifier(iterations=220, depth=3,
            learning_rate=0.035, l2_leaf_reg=5, random_seed=42,
            thread_count=1, verbose=False, allow_writing_files=False)),
    ])
    return make_pipeline(preprocess, ensemble)

print(f"Таксономических групп: {len(motu_groups)}. Статистических признаков: {len(stat_feature_names)}.")

Таксономических групп: 124. Статистических признаков: 744.


## Проверка качества

Пять повторов пятикратной стратифицированной проверки и пять проверок с исключением целой когорты (LOCO). Считаем ROC AUC по вероятностям и сбалансированную точность при пороге 0.50. Выбор оральных маркеров был связан с анализом меток всех 415 участников, поэтому результаты остаются исследовательскими.

In [4]:
strata = dev.cohort + "/" + dev.label.astype(str)
folds = []
for seed in (42, 17, 73, 101, 211):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (train, valid) in enumerate(cv.split(dev, strata), 1):
        folds.append(("CV", seed, str(fold), train, valid))
for train, valid in LeaveOneGroupOut().split(dev, groups=dev.cohort):
    folds.append(("LOCO", 0, dev.cohort.iloc[valid[0]], train, valid))

rows = []
with threadpool_limits(limits=1):
    for scheme, seed, fold, train, valid in folds:
        model = make_model().fit(dev.iloc[train], dev.label.iloc[train])
        p = model.predict_proba(dev.iloc[valid])[:, 1]
        target = dev.label.iloc[valid]
        rows.append((scheme, seed, fold, roc_auc_score(target, p),
                     balanced_accuracy_score(target, p >= 0.50)))
scores = pd.DataFrame(rows, columns=["scheme", "seed", "fold", "auc", "ba"])
display(scores.groupby("scheme")[["auc", "ba"]].agg(["mean", "std"]).round(4))

auc              ba        
         mean     std    mean     std
scheme                               
CV      0.904  0.0339  0.8123  0.0383
LOCO    0.866  0.0488  0.7952  0.0535

## Предсказания для новых образцов

Если test.csv доступен, ансамбль переобучается на всех размеченных строках. В submission_oral_ensemble_B.csv сохраняются sample_id и непрерывная оценка label в исходном порядке. Последняя ячейка показывает метки 0/1 при пороге 0,5.

In [5]:
PREDICT_TEST = TEST.is_file()
if PREDICT_TEST:
    full_train = pd.read_csv(TRAIN, dtype={"sample_id": str}).sort_values("sample_id")
    test = pd.read_csv(TEST, dtype={"sample_id": str})
    assert full_train.sample_id.notna().all() and full_train.sample_id.is_unique
    assert test.sample_id.notna().all() and test.sample_id.is_unique
    assert not test.sample_id.isin(full_train.sample_id).any()
    assert set(test.columns) == set(full_train.columns) - {"label"}
    if SUBMISSION.resolve() in {TRAIN.resolve(), TEST.resolve()}:
        raise ValueError("Путь для CSV совпадает с входным файлом")
    with threadpool_limits(limits=1):
        final_model = make_model().fit(full_train, full_train.label)
        probabilities = final_model.predict_proba(test)[:, 1]
        reordered = final_model.predict_proba(test.iloc[::3, ::-1])[:, 1]
    np.testing.assert_allclose(reordered, probabilities[::3], rtol=0, atol=1e-12)
    assert np.isfinite(probabilities).all() and ((0 <= probabilities) & (probabilities <= 1)).all()
    submission = pd.DataFrame({"sample_id": test.sample_id.to_numpy(), "label": probabilities})
    submission.to_csv(SUBMISSION, index=False, lineterminator="\n")
    labels = submission.assign(label=submission.label.ge(0.5).astype(int))
    with pd.option_context("display.max_rows", None):
        display(labels)
else:
    print("Для сохранения CSV укажите путь к test.csv в GUT_TEST.")


,sample_id,label
0,S0125,0
1,S0242,0
2,S0053,0
3,S0173,0
4,S0453,0
5,S0342,0
6,S0438,0
7,S0222,0
8,S0429,0
9,S0134,0
